# MarineMamba GPU Runner

Runs all GPU experiments. Upload your `data/processed/` folder to Google Drive first.

- **Models C, D, E**: Use T4 (free tier)
- **Model F (Evo 2)**: Use A100 (Colab Pro)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kluless13/marinemamba/blob/main/notebooks/gpu_runner.ipynb)

In [ ]:
# === SETUP ===
!pip install lightning einops timm torchtext transformers scikit-learn pandas numpy tqdm huggingface_hub

# Install mamba-ssm with pre-built wheels (avoids CUDA build issues)
!pip install mamba-ssm --no-build-isolation
!pip install causal-conv1d --no-build-isolation

# If above fails, try pinned versions:
# !pip install mamba-ssm==2.2.1 causal-conv1d==1.3.0.post1 --no-build-isolation

!git clone https://github.com/kluless13/marinemamba.git
%cd marinemamba

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Copy data from Drive to local (faster I/O)
!cp -r /content/drive/MyDrive/marinemamba/data/processed data/processed

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Model C: Direct Transfer (insect → fish) — ~1 hour on T4

In [ ]:
!python scripts/04_barcodemamba_models.py --mode transfer --data-dir data/processed --output-dir results

## Model E: Domain Adaptation (insect → fish pretrain → fine-tune) — ~8-12 hours on T4

In [ ]:
!python scripts/04_barcodemamba_models.py --mode adapt --data-dir data/processed --output-dir results

## Model D: Scratch Pretrain — ~12-20 hours on T4 (run overnight)

In [ ]:
!python scripts/04_barcodemamba_models.py --mode scratch --data-dir data/processed --output-dir results

## Model F: Evo 2 Embeddings — ~2-3 hours on A100 (requires Colab Pro)

**Switch runtime to A100 before running this cell.**

In [ ]:
# Evo 2 needs additional deps
!pip install evo2 flash-attn --no-build-isolation
!python scripts/05_evo2_embeddings.py --data-dir data/processed --output-dir results

## Save Results to Drive

In [ ]:
!cp -r results/ /content/drive/MyDrive/marinemamba/results/
!cp -r checkpoints/ /content/drive/MyDrive/marinemamba/checkpoints/
print("Results saved to Google Drive")

# Show all results
import json, glob
for f in sorted(glob.glob("results/*.json")):
    print(f"\n{'=' * 60}")
    print(f)
    print('=' * 60)
    print(json.dumps(json.load(open(f)), indent=2))